# 5D chaotic model — corrected image encryption and decryption

**Corrected against the 2024 paper's 5D model equation.** This notebook implements the **classical case** \(\alpha=1\) using RK4. It does **not** implement the paper's Atangana–Baleanu–Caputo (ABC) fractional numerical scheme for \(\alpha=0.99\).

The first model equation is implemented as

\[
\dot{x}_1=a(x_2-x_1)+0.25x_3+g\left(\delta+\gamma\tanh(x_5)\right)x_3.
\]

The previous notebook incorrectly used \(\tanh(\gamma x_5)\). The placement of \(\gamma\) matters and has been corrected here.

The image pipeline uses the three flow variables \(x_1,x_2,x_3\) to make byte keystreams and XORs them with the RGB image and, optionally, a second RGB key image. The particular conversion from floating-point orbit values to bytes is an implementation choice; it is not specified as a formula in the paper. This is a reversible experimental implementation, **not a claim of proven cryptographic security**. In particular, XOR-only encryption has no diffusion, so one-pixel differential NPCR/UACI results are expected to be small.


## Step 1: Imports

In [ ]:
import io, hashlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

# numba makes the chaotic solver fast. If it is missing, plain Python is used (slower, same result).
try:
    from numba import njit
except ImportError:
    def njit(f):
        return f

## Step 2: The corrected 5D chaotic model (paper Eq. 7 / Eq. 14)

State variables are \(x_1,\ldots,x_5\). The parameters are `p = [a, b, c, d, e, g, delta, gamma]`.

The corrected first equation contains

\[
g\left(\delta+\gamma\tanh(x_5)\right)x_3,
\]

not `g * (delta + tanh(gamma*x5)) * x3`.

We solve this with classic RK4, which corresponds to the **integer-order case \(\alpha=1\)**. The paper's ABC fractional scheme is required to reproduce its \(\alpha=0.99\) simulations.

In [ ]:
@njit
def model(s, p):
    x1, x2, x3, x4, x5 = s[0], s[1], s[2], s[3], s[4]
    a, b, c, d, e, g, delta, gamma = p[0], p[1], p[2], p[3], p[4], p[5], p[6], p[7]
    out = np.empty(5)
    out[0] = a * (x2 - x1) + 0.25 * x3 + g * (delta + gamma * np.tanh(x5)) * x3  # corrected paper term: g*(delta + gamma*tanh(x5))*x3
    out[1] = b * x1 - x1 * x3 + c * x2 - x4
    out[2] = d * x1 * x2 - e * x3
    out[3] = x1
    out[4] = x3
    return out

@njit
def rk4_step(s, p, h):
    k1 = model(s, p)
    k2 = model(s + 0.5 * h * k1, p)
    k3 = model(s + 0.5 * h * k2, p)
    k4 = model(s + h * k3, p)
    return s + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

@njit
def generate_orbit(x0, p, h, n_transient, n_samples, skip):
    s = x0.copy()
    for _ in range(n_transient):          # throw away the start (system settles on the attractor)
        s = rk4_step(s, p, h)
    out = np.empty((n_samples, 3))
    for i in range(n_samples):
        for _ in range(skip):             # take one sample every `skip` steps
            s = rk4_step(s, p, h)
        out[i, 0] = s[0]
        out[i, 1] = s[1]
        out[i, 2] = s[2]
    return out

## Step 3: Secret key

For reproducibility, the defaults below use the model parameters and initial conditions stated in the paper. The exact initial conditions are part of the deterministic keystream key. For a demonstration with a different secret key, change the initial conditions and use exactly the same values during decryption.

**Security note:** the paper's published parameters and initial conditions are not secret. Do not describe the default published values as a secure private key.

In [ ]:
KEY = dict(
    a=78, b=-16, c=58, d=35, e=15, g=1, delta=1, gamma=0.5,   # system parameters (paper values)
    x0=[0.0, 0.0, 8.0, 6.0, 0.0],                             # initial conditions: CHANGE THESE
)
H_STEP    = 0.001   # RK4 step size
TRANSIENT = 10000   # steps thrown away at the start
SKIP      = 10      # steps between two samples

def key_to_arrays(key):
    p  = np.array([key['a'], key['b'], key['c'], key['d'],
                   key['e'], key['g'], key['delta'], key['gamma']], dtype=np.float64)
    x0 = np.array(key['x0'], dtype=np.float64)
    return x0, p

## Step 4: Encryption and decryption (AIE structure described in the paper)

The RGB image is processed channel-wise. The chaotic orbit supplies three streams: \(x_1\) for R, \(x_2\) for G and \(x_3\) for B. Each channel is XORed with its chaotic byte stream and, when enabled, with the corresponding channel of a second key image.

\[
C=P\oplus K_{chaos}\oplus K_{image},\qquad
P=C\oplus K_{chaos}\oplus K_{image}.
\]

XOR is its own inverse, so the exact same chaotic key and key image are required for recovery. The byte-quantization step used in this notebook is an implementation detail; the paper does not prescribe this exact Python conversion formula.

In [ ]:
def make_keystream(shape, key):
    """Generate HxWx3 uint8 keystreams from chaotic states x1, x2 and x3."""
    if len(shape) != 3 or shape[2] != 3:
        raise ValueError("Expected an RGB image with shape (height, width, 3).")
    H, W, _ = shape
    if H < 1 or W < 1:
        raise ValueError("Image dimensions must be non-zero.")
    L = H * W
    x0, p = key_to_arrays(key)
    if x0.shape != (5,) or p.shape != (8,):
        raise ValueError("The key must contain five initial conditions and eight model parameters.")
    orbit = generate_orbit(x0, p, H_STEP, TRANSIENT, L, SKIP)
    if not np.all(np.isfinite(orbit)):
        raise ValueError("The chaotic system produced non-finite values. Check the key and integration settings.")
    # Implementation-specific float-to-byte conversion (not a formula stated by the paper).
    ks = (np.floor(np.abs(orbit) * 1e8).astype(np.int64) % 256).astype(np.uint8)
    return ks.reshape(H, W, 3)


def _validate_rgb_uint8(image, name):
    arr = np.asarray(image)
    if arr.ndim != 3 or arr.shape[2] != 3:
        raise ValueError(f"{name} must be an RGB array with shape (height, width, 3).")
    if arr.dtype != np.uint8:
        raise TypeError(f"{name} must have dtype uint8; received {arr.dtype}.")
    return arr


def _validate_key_image(key_img, shape):
    if key_img is None:
        return None
    key_img = _validate_rgb_uint8(key_img, "key_img")
    if key_img.shape != shape:
        raise ValueError(f"key_img shape {key_img.shape} must match image shape {shape}.")
    return key_img


def encrypt_image(img, key, key_img=None):
    """Encrypt an RGB uint8 image using chaotic XOR and optional RGB image XOR."""
    img = _validate_rgb_uint8(img, "img")
    key_img = _validate_key_image(key_img, img.shape)
    cipher = np.bitwise_xor(img, make_keystream(img.shape, key))
    if key_img is not None:
        cipher = np.bitwise_xor(cipher, key_img)
    return cipher


def decrypt_image(cipher, key, key_img=None):
    """Decrypt an RGB uint8 ciphertext using the same key material."""
    cipher = _validate_rgb_uint8(cipher, "cipher")
    key_img = _validate_key_image(key_img, cipher.shape)
    plain = np.bitwise_xor(cipher, make_keystream(cipher.shape, key))
    if key_img is not None:
        plain = np.bitwise_xor(plain, key_img)
    return plain


## Step 5: Upload your plain image

Choose an RGB image. Set `RESIZE_TO` to a number (for example, `256`) to resize to a square for a quick demonstration, or leave it `None` to keep the original dimensions. Any resizing changes the data being encrypted; decryption will recover the resized working image, not the pre-resize source file.

In [ ]:
RESIZE_TO = None     # e.g. 256, or None to keep the original size

def upload_image(resize_to=None, size=None):
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    im = Image.open(io.BytesIO(uploaded[name])).convert('RGB')   # alpha channel (if any) is dropped
    if size is not None:
        im = im.resize(size)
    elif resize_to is not None:
        im = im.resize((resize_to, resize_to))
    return np.array(im, dtype=np.uint8), name

plain, plain_name = upload_image(RESIZE_TO)
print("Loaded:", plain_name, "| size (H, W, channels):", plain.shape)
plt.imshow(plain); plt.title("Plain image"); plt.axis('off'); plt.show()

## Step 6 (optional): Upload a key image

The paper describes use of the chaotic 3D flow together with the plain image/key-image material. This notebook optionally uses a second RGB image as an additional XOR key. Decryption requires the exact same key image bytes and dimensions. Set `USE_KEY_IMAGE = False` to run without the image key.

In [ ]:
USE_KEY_IMAGE = True

key_img = None
if USE_KEY_IMAGE:
    key_img, key_name = upload_image(size=(plain.shape[1], plain.shape[0]))   # auto-resized to match
    print("Key image:", key_name)
    plt.imshow(key_img); plt.title("Key image"); plt.axis('off'); plt.show()
else:
    print("No key image used.")

## Step 7: Encrypt

The encrypted output is saved as PNG, a lossless format. Keep the key parameters, initial conditions and (when enabled) the exact key image to decrypt it.

In [ ]:
cipher = encrypt_image(plain, KEY, key_img)

plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1); plt.imshow(plain);  plt.title("Plain");     plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(cipher); plt.title("Encrypted"); plt.axis('off')
plt.show()

Image.fromarray(cipher).save("encrypted.png")      # PNG = lossless
files.download("encrypted.png")

## Step 8: Decrypt

- `LOAD_ENCRYPTED_FROM_FILE = False` decrypts the encrypted image already in memory.
- Set it to `True` to upload a previously saved encrypted PNG. Keep the same `KEY` and, if used, the same key image.

The equality check below verifies exact recovery of the working RGB array.

In [ ]:
LOAD_ENCRYPTED_FROM_FILE = False

if LOAD_ENCRYPTED_FROM_FILE:
    cipher_in, _ = upload_image()
else:
    cipher_in = cipher

decrypted = decrypt_image(cipher_in, KEY, key_img)

plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1); plt.imshow(cipher_in);  plt.title("Encrypted"); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(decrypted);  plt.title("Decrypted"); plt.axis('off')
plt.show()

if decrypted.shape == plain.shape:
    print("Decrypted == original (every pixel identical):", np.array_equal(decrypted, plain))
Image.fromarray(decrypted).save("decrypted.png")
files.download("decrypted.png")

## Step 9: Image statistics and correctness checks

The paper discusses histogram, entropy, correlation, MSE/PSNR, NPCR and UACI. This notebook reports empirical histogram/entropy/correlation, then verifies exact correct-key recovery and tests wrong keys.

**Important distinction for NPCR/UACI:** the implementation here is an XOR mask with no diffusion. A one-pixel plaintext change under the same keystream changes only that pixel. Therefore, a low NPCR/UACI in the explicit one-pixel differential experiment is mathematically expected; it must not be forced to match a generic 99% / 33% target. The paper's Table 2 reports its own values (NPCR 99.6261% and UACI 29.889% for Baboon I); this notebook's differential protocol is not a direct reproduction of that table. A small differential result identifies a limitation of this XOR-only implementation.

In [ ]:
def entropy(channel):
    counts = np.bincount(channel.ravel(), minlength=256).astype(np.float64)
    probs = counts[counts > 0] / channel.size
    return float(-np.sum(probs * np.log2(probs)))


def adjacent_correlation(channel, n=5000, seed=0):
    """Sample horizontal neighbours; returns NaN for images too small to test."""
    if channel.ndim != 2 or channel.shape[0] < 1 or channel.shape[1] < 2:
        return float('nan')
    rng = np.random.default_rng(seed)
    rows = rng.integers(0, channel.shape[0], n)
    cols = rng.integers(0, channel.shape[1] - 1, n)
    left = channel[rows, cols].astype(np.float64)
    right = channel[rows, cols + 1].astype(np.float64)
    if np.std(left) == 0 or np.std(right) == 0:
        return float('nan')
    return float(np.corrcoef(left, right)[0, 1])


def mse(a, b):
    a = np.asarray(a).astype(np.float64)
    b = np.asarray(b).astype(np.float64)
    if a.shape != b.shape:
        raise ValueError("MSE requires arrays with identical shapes.")
    return float(np.mean((a - b) ** 2))


def psnr(a, b):
    value = mse(a, b)
    return float('inf') if value == 0 else float(10 * np.log10((255.0 ** 2) / value))


def differential_npcr_uaci(cipher_a, cipher_b):
    """Compare two ciphertexts generated from a one-pixel plaintext change."""
    a = _validate_rgb_uint8(cipher_a, "cipher_a")
    b = _validate_rgb_uint8(cipher_b, "cipher_b")
    if a.shape != b.shape:
        raise ValueError("Ciphertexts must have the same shape.")
    changed_pixels = np.any(a != b, axis=2)
    npcr = float(np.mean(changed_pixels) * 100.0)
    uaci = float(np.mean(np.abs(a.astype(np.float64) - b.astype(np.float64)) / 255.0) * 100.0)
    return npcr, uaci


# 1) Histograms: encrypted channels should look more evenly distributed than structured plain-image channels.
plt.figure(figsize=(11, 3.8))
plt.subplot(1, 2, 1)
plt.hist(plain.ravel(), bins=256, range=(0, 255))
plt.title("Histogram: plain RGB values")
plt.subplot(1, 2, 2)
plt.hist(cipher.ravel(), bins=256, range=(0, 255))
plt.title("Histogram: encrypted RGB values")
plt.tight_layout()
plt.show()

# 2) Entropy and horizontal adjacent-pixel correlation, channel by channel.
plain_entropy = [entropy(plain[..., i]) for i in range(3)]
cipher_entropy = [entropy(cipher[..., i]) for i in range(3)]
plain_corr = [adjacent_correlation(plain[..., i], seed=i) for i in range(3)]
cipher_corr = [adjacent_correlation(cipher[..., i], seed=i) for i in range(3)]
print("Shannon entropy (maximum 8 bits)")
print("  Plain RGB     :", [round(v, 4) for v in plain_entropy])
print("  Encrypted RGB :", [round(v, 4) for v in cipher_entropy])
print("Horizontal adjacent correlation")
print("  Plain RGB     :", [round(v, 4) if np.isfinite(v) else v for v in plain_corr])
print("  Encrypted RGB :", [round(v, 4) if np.isfinite(v) else v for v in cipher_corr])

# 3) Correct key: exact equality, MSE=0 and PSNR=infinity are the expected round-trip results.
dec_ok = decrypt_image(cipher, KEY, key_img)
exact_recovery = np.array_equal(dec_ok, plain)
print("\nCorrect-key decryption")
print("  Exact array recovery:", exact_recovery)
print("  MSE :", mse(dec_ok, plain))
print("  PSNR:", psnr(dec_ok, plain), "dB (infinity is expected for exact equality)")

# 4) One-pixel differential test: XOR-only encryption has no diffusion.
# Change one complete RGB pixel and encrypt it again with the same deterministic key material.
plain_changed = plain.copy()
rr, cc = plain.shape[0] // 2, plain.shape[1] // 2
plain_changed[rr, cc, :] = (plain_changed[rr, cc, :].astype(np.uint16) + 1).astype(np.uint8)
cipher_changed = encrypt_image(plain_changed, KEY, key_img)
d_npcr, d_uaci = differential_npcr_uaci(cipher, cipher_changed)
expected_one_pixel_percent = 100.0 / (plain.shape[0] * plain.shape[1])
print("\nOne-pixel differential test (same key; XOR-only, no diffusion)")
print(f"  Measured NPCR: {d_npcr:.8f}%")
print(f"  Measured UACI: {d_uaci:.8f}%")
print(f"  One-pixel NPCR expected from this construction: {expected_one_pixel_percent:.8f}%")
print(f"  UACI upper bound for this one-pixel RGB change: {expected_one_pixel_percent:.8f}%")
print("  Interpretation: low values are expected here; this is not a diffusion-based cipher.")

# 5) Wrong chaotic key: a tiny x1(0) perturbation should generally prevent correct recovery.
bad_key = dict(KEY)
bad_key['x0'] = list(KEY['x0'])
bad_key['x0'][0] += 1e-10
wrong_chaotic = decrypt_image(cipher, bad_key, key_img)
print("\nWrong chaotic key (x1(0) shifted by 1e-10)")
print("  Exact recovery:", np.array_equal(wrong_chaotic, plain))
print(f"  MSE: {mse(wrong_chaotic, plain):.4f}")
print(f"  PSNR: {psnr(wrong_chaotic, plain):.4f} dB")

# 6) Wrong key image: invert the supplied image-key bytes to create a deterministic incorrect key image.
if key_img is not None:
    wrong_key_image = np.bitwise_not(key_img)
    wrong_image_decryption = decrypt_image(cipher, KEY, wrong_key_image)
    print("\nWrong key image (bitwise-inverted key image)")
    print("  Exact recovery:", np.array_equal(wrong_image_decryption, plain))
    print(f"  MSE: {mse(wrong_image_decryption, plain):.4f}")
    print(f"  PSNR: {psnr(wrong_image_decryption, plain):.4f} dB")
else:
    print("\nWrong key-image test: SKIPPED because USE_KEY_IMAGE=False.")

# 7) Reproducibility fingerprint for the first 1000 generated RGB keystream triplets.
ks_check = make_keystream(plain.shape, KEY)
fingerprint = hashlib.sha256(ks_check.reshape(-1, 3)[:1000].tobytes()).hexdigest()[:16]
print("\nKeystream fingerprint (same key/settings should reproduce this):", fingerprint)

# Paper comparison note.
print("\nPaper comparison:")
print("  The paper reports Baboon-I NPCR=99.6261% and UACI=29.889% in its Table 2.")
print("  This notebook's one-pixel differential protocol is different and should not be presented as reproducing those table values.")
print("  The corrected ODE here is the alpha=1 classical case; ABC fractional alpha=0.99 is not implemented in this notebook.")
